# ClarioAI — Agent Test Notebook (Refactored Schema)

Notebook ini menguji **agent nodes saja** dengan schema baru (structured outputs).

| # | Seksi | Deskripsi |
|---|-------|-----------|
| 1 | Imports & Environment | Verifikasi semua package tersedia |
| 2 | Schema Baru | Verifikasi dataclass baru + EBPState |
| 3 | Sample State | Helper `make_fresh_state` dengan schema baru |
| 4 | Market Scout | Output structured `MarketScoutReport` |
| 5 | Strategic Architect | Output structured `StrategicReport` |
| 6 | Financial Analyst | Output structured `FinancialAnalysisReport` (int IDR) |
| 7 | Ethics Agent | Output structured `EthicsAnalysisReport` |
| 8 | Validation Layer | Rule-based contradiction detection (no LLM) |
| 9 | Lead Orchestrator | Verdict engine — bukan essay |
| 10 | Ringkasan Hasil | Print semua fields terukur dari tiap agent |

---
## 1. Imports & Environment Check

In [1]:
import sys, importlib

REQUIRED_PACKAGES = [
    "langchain",
    "langchain_openai",
    "langchain_core",
    "langgraph",
    "requests",
]

missing = []
for pkg in REQUIRED_PACKAGES:
    try:
        importlib.import_module(pkg)
        print(f"  [OK] {pkg}")
    except ImportError:
        print(f"  [MISSING] {pkg}")
        missing.append(pkg)

if missing:
    print(f"\nInstall missing: pip install {' '.join(missing)}")
else:
    print("\nSemua package tersedia.")

  [OK] langchain
  [OK] langchain_openai
  [OK] langchain_core
  [OK] langgraph
  [OK] requests

Semua package tersedia.


In [2]:
import os
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

Project root: d:\Gabriel\Gabriel\Telkom University\Kuliah\Semester VI\Capstone\Multi-Agent-Decision-Support-System


In [3]:
# ── Monkey-patch format_constraints ──
# Ini menggantikan sementara format_constraints lama (yang pakai field schema lama)
# dengan versi baru yang support schema baru.
# Cara permanen: edit fungsi format_constraints di functions/agent_utils.py

import functions.agent_utils as _au

def _format_constraints_new(bc) -> str:
    """format_constraints updated untuk schema baru BusinessConstraints."""
    if bc is None:
        return "No constraints provided."
    lines = []
    if hasattr(bc, 'sector'):
        lines.append(f"Sector: {bc.sector}")
    elif hasattr(bc, 'sector_and_domain'):
        lines.append(f"Sector/Domain: {bc.sector_and_domain}")
    if hasattr(bc, 'target_audience'):
        lines.append(f"Target Audience: {bc.target_audience}")
    elif hasattr(bc, 'audience'):
        lines.append(f"Target Audience: {bc.audience}")
    if hasattr(bc, 'business_idea'):
        lines.append(f"Business Idea: {bc.business_idea}")
    elif hasattr(bc, 'initial_prompt'):
        lines.append(f"Business Idea: {bc.initial_prompt}")
    if hasattr(bc, 'location'):
        lines.append(f"Location: {bc.location}")
    if hasattr(bc, 'budget_range'):
        lines.append(f"Budget Range: {bc.budget_range}")
    if hasattr(bc, 'business_model'):
        lines.append(f"Business Model: {bc.business_model}")
    if hasattr(bc, 'experience_level'):
        lines.append(f"Experience Level: {bc.experience_level}")
    if hasattr(bc, 'risk_tolerance'):
        lines.append(f"Risk Tolerance: {bc.risk_tolerance}")
    return "\n".join(lines)

_au.format_constraints = _format_constraints_new
print("format_constraints monkey-patched: OK")
print("Test:", _au.format_constraints(None))

format_constraints monkey-patched: OK
Test: No constraints provided.


---
## 2. Verifikasi Schema Baru

In [4]:
# Import semua class dari schema baru
from states.schema import (
    BusinessConstraints,
    CompetitorInsight,
    MarketScoutReport,
    MonthlyMilestone,
    StrategicReport,
    SensitivityScenario,
    FinancialAnalysisReport,
    EthicsAnalysisReport,
    ContradictionFlag,
    ValidationReport,
    EBPState,
)
print("Import states.schema (refactored): OK")

Import states.schema (refactored): OK


In [5]:
# --- Test BusinessConstraints (schema baru — field lengkap) ---
constraints = BusinessConstraints(
    sector="F&B / Kuliner",
    target_audience="Mahasiswa dan pekerja kantoran usia 18-35 di Bandung",
    business_idea="Kedai kopi spesialti dengan konsep coworking space, menyasar remote worker dan mahasiswa",
    location="Bandung, Jawa Barat",
    budget_range="Rp 50.000.000 – Rp 150.000.000",
    business_model="offline",
    experience_level="beginner",
    risk_tolerance="medium",
)

print("BusinessConstraints:")
print(f"  sector          : {constraints.sector}")
print(f"  business_idea   : {constraints.business_idea}")
print(f"  location        : {constraints.location}")
print(f"  budget_range    : {constraints.budget_range}")
print(f"  business_model  : {constraints.business_model}")
print(f"  experience_level: {constraints.experience_level}")
print(f"  risk_tolerance  : {constraints.risk_tolerance}")

BusinessConstraints:
  sector          : F&B / Kuliner
  business_idea   : Kedai kopi spesialti dengan konsep coworking space, menyasar remote worker dan mahasiswa
  location        : Bandung, Jawa Barat
  budget_range    : Rp 50.000.000 – Rp 150.000.000
  business_model  : offline
  experience_level: beginner
  risk_tolerance  : medium


In [6]:
# --- Test instantiate semua dataclass baru ---
import uuid

# MarketScoutReport
sample_market = MarketScoutReport(
    demand_score=7,
    competition_level="high",
    opportunity_score=6,
    market_trend_summary="Pasar kopi spesialti tumbuh 12% YoY di kota tier-1 Indonesia.",
    customer_pain_points=["Harga kopi mahal", "Tidak ada tempat kerja nyaman", "WiFi lambat di kafe biasa"],
    competitors=[
        CompetitorInsight(
            name="Kopi Kenangan",
            strengths=["Brand awareness tinggi", "Harga terjangkau"],
            weaknesses=["Tidak ada coworking space", "Antrian panjang"],
            market_position="Market leader F&B kopi grab-and-go",
        )
    ],
    tam_estimate="Rp 12 Triliun/tahun",
    sam_estimate="Rp 800 Miliar/tahun",
    som_estimate="Rp 2 Miliar/tahun (target 12 bulan)",
    market_sizing_methodology="Populasi mahasiswa+pekerja Bandung 1,2 juta × Rp 50.000/bulan spending kopi",
    validated_opportunities=[
        "Kopi spesialti + coworking — gap tidak ada pemain yang gabungkan keduanya di Bandung",
        "Paket day-pass coworking Rp 50.000 termasuk 1 kopi",
    ],
    source_links=["https://example.com/edtech-market"],
    confidence_score=7,
)

# StrategicReport
sample_strategy = StrategicReport(
    strengths=["Konsep unik coworking+kopi belum ada di Bandung", "Harga kompetitif vs warung kopi premium"],
    weaknesses=["Founder tidak berpengalaman F&B", "Modal terbatas untuk lokasi premium"],
    opportunities=["Tren remote work pasca-pandemi masih tinggi", "Demand mahasiswa untuk tempat belajar nyaman"],
    threats=["Kopi Kenangan bisa buka outlet coworking", "Kenaikan harga biji kopi global"],
    pestel_highlights=[
        "Political: Kebijakan UMKM Bandung mendukung usaha kecil",
        "Economic: Inflasi 4,2% pengaruhi biaya operasional",
        "Social: Gen Z dan millennial prefer third place work environment",
        "Legal: Wajib NIB + izin Pemda untuk usaha F&B",
    ],
    top_problem_priorities=[
        "Validasi lokasi strategis dengan traffic tinggi sebelum tanda tangan lease",
        "Rekrut barista berpengalaman sebelum launch",
    ],
    strategic_order_of_operations="Amankan lokasi → rekrut barista → soft launch dengan promo → baru scale marketing.",
    roadmap={
        "month_1": MonthlyMilestone(
            objective="Setup operasional dan soft launch",
            kpis=["50 pelanggan/hari di minggu ke-4", "Rating Google Maps minimal 4.3"],
        ),
        "month_2": MonthlyMilestone(
            objective="Capai BEP mingguan",
            kpis=["Revenue Rp 30 juta/bulan", "30 member day-pass coworking"],
        ),
        "month_3": MonthlyMilestone(
            objective="Scale melalui digital marketing",
            kpis=["100 followers IG/minggu", "Revenue Rp 50 juta/bulan"],
        ),
    },
    value_proposition="Satu-satunya kedai kopi spesialti di Bandung yang menawarkan meja coworking nyaman dengan WiFi 100 Mbps.",
    customer_segments="Mahasiswa PTN/PTS Bandung dan remote worker usia 22-35 yang butuh third place produktif.",
    channels="Instagram, Google Maps, word-of-mouth, partnership dengan komunitas remote worker Bandung.",
    cost_structure="Sewa lokasi (40%), bahan baku kopi (25%), gaji 2 barista (20%), operasional (15%).",
    revenue_streams="Penjualan kopi (60%), day-pass coworking Rp 50.000 (30%), merchandise (10%).",
    recommended_sales_channel="Walk-in + GrabFood/GoFood untuk takeaway",
    confidence_score=7,
)

# FinancialAnalysisReport
sample_financial = FinancialAnalysisReport(
    estimated_monthly_revenue_rp=45_000_000,
    estimated_monthly_cogs_rp=13_500_000,
    estimated_monthly_fixed_costs_rp=22_000_000,
    estimated_monthly_net_profit_rp=9_500_000,
    bep_units_per_month=440,
    bep_revenue_rp=22_000_000,
    cash_runway_months=8,
    estimated_startup_cost_rp=120_000_000,
    estimated_break_even_months=13,
    sensitivity_scenarios=[
        SensitivityScenario(label="Volume penjualan turun 20%", net_profit_rp=200_000),
        SensitivityScenario(label="COGS naik 15%", net_profit_rp=7_477_500),
        SensitivityScenario(label="Marketing cost naik 50%", net_profit_rp=6_500_000),
    ],
    industry_benchmark_notes="Gross margin 60% sesuai benchmark F&B kafe. Break-even 13 bulan di atas rata-rata industri 12 bulan.",
    financial_feasibility_verdict="LAYAK_DENGAN_CATATAN",
    key_financial_risks=["Sewa lokasi naik setelah kontrak awal", "Volume bulan pertama di bawah estimasi"],
    assumptions=[
        "Diasumsikan 50 pelanggan/hari rata-rata dengan AOV Rp 35.000",
        "Sewa lokasi Rp 12 juta/bulan berdasarkan benchmark area Dago-Dipati Ukur",
        "2 barista dengan gaji masing-masing Rp 4 juta/bulan",
    ],
    affordability_score=6,
    confidence_score=7,
)

# EthicsAnalysisReport
sample_ethics = EthicsAnalysisReport(
    mandatory_permits=["NIB via OSS", "NPWP Perorangan/Badan"],
    sector_specific_permits=["Sertifikasi Halal BPJPH (untuk minuman/makanan)", "PIRT dari Dinas Kesehatan (jika produksi sendiri)"],
    compliance_checklist_perizinan=["Daftar OSS dan dapatkan NIB", "Ajukan izin lokasi ke Pemda Bandung"],
    compliance_checklist_pajak=["Daftar NPWP", "Laporkan PPh 21 jika ada karyawan"],
    compliance_checklist_legalitas=["Buat perjanjian kerja tertulis untuk barista", "Pasang informasi harga yang jelas"],
    critical_risk_alerts=[
        "Beroperasi tanpa NIB: denda administratif hingga Rp 5 juta (PP 5/2021)",
        "Menjual produk kopi tanpa label yang benar: sanksi UU Pangan No. 18/2012",
    ],
    legal_risk_level="low",
    bureaucracy_estimated_duration="2–3 minggu untuk NIB + izin dasar",
    bureaucracy_complexity="RENDAH",
    compliance_score=8,
    confidence_score=8,
)

print("Semua sample dataclass baru: OK")
print(f"  MarketScoutReport.demand_score     : {sample_market.demand_score}")
print(f"  StrategicReport.strengths count    : {len(sample_strategy.strengths)}")
print(f"  FinancialAnalysisReport.net_profit : Rp {sample_financial.estimated_monthly_net_profit_rp:,}")
print(f"  EthicsAnalysisReport.legal_risk    : {sample_ethics.legal_risk_level}")

Semua sample dataclass baru: OK
  MarketScoutReport.demand_score     : 7
  StrategicReport.strengths count    : 2
  FinancialAnalysisReport.net_profit : Rp 9,500,000
  EthicsAnalysisReport.legal_risk    : low


In [7]:
# --- Test EBPState instantiation dengan schema baru ---
sample_state: EBPState = {
    "state_id": str(uuid.uuid4()),
    "business_constraints": constraints,    # key baru (bukan 'bussiness_constraints')
    "market_report": None,                  # key baru (bukan 'market_scout_report')
    "strategy_report": None,                # key baru (bukan 'strategic_report' lama)
    "financial_report": None,               # key baru (bukan 'financial_analysis_report')
    "ethics_report": None,                  # key baru (bukan 'ethics_analysis_report')
    "validation_report": None,              # NEW field
    "approval_status": "pending",
    "orchestrator_feedback": None,
    "confidence_score": 0,
    "final_verdict": None,
    "final_recommendations": [],
    "messages": [],
    "iteration": 0,
    "max_iterations": 3,
    "user_feedback": None,
    "final_result": None,
}

print("EBPState (schema baru):")
print(f"  state_id         : {sample_state['state_id']}")
print(f"  approval_status  : {sample_state['approval_status']}")
print(f"  market_report    : {sample_state['market_report']} (None = belum dijalankan)")
print(f"  validation_report: {sample_state['validation_report']} (None = belum dijalankan)")

EBPState (schema baru):
  state_id         : 41a5a589-1420-4ea5-8191-a13be6b57965
  approval_status  : pending
  market_report    : None (None = belum dijalankan)
  validation_report: None (None = belum dijalankan)


---
## 3. Helper: make_fresh_state

In [8]:
# Storage untuk hasil agent — diisi setelah setiap agent selesai
# Dipakai oleh make_fresh_state agar agent berikutnya pakai hasil nyata
_agent_results = {
    "report_ms": None,   # hasil Market Scout nyata
    "report_sa": None,   # hasil Strategic Architect nyata
    "report_fa": None,   # hasil Financial Analyst nyata
    "report_ea": None,   # hasil Ethics Agent nyata
    "report_vl": None,   # hasil Validation Layer nyata
}

def make_fresh_state(
    iteration: int = 0,
    with_market: bool = False,
    with_strategy: bool = False,
    with_financial: bool = False,
    with_ethics: bool = False,
    with_validation: bool = False,
    orchestrator_feedback: str | None = None,
) -> EBPState:
    """Buat state bersih untuk test tiap agent.
    
    Gunakan hasil nyata dari agent sebelumnya jika tersedia di _agent_results,
    fallback ke sample data jika belum ada.
    """
    # Gunakan hasil nyata jika tersedia, fallback ke sample
    mkt  = (_agent_results["report_ms"] or sample_market)   if with_market    else None
    strat = (_agent_results["report_sa"] or sample_strategy) if with_strategy  else None
    fin   = (_agent_results["report_fa"] or sample_financial) if with_financial else None
    eth   = (_agent_results["report_ea"] or sample_ethics)   if with_ethics    else None
    val   = _agent_results["report_vl"]                       if with_validation else None

    state: EBPState = {
        "state_id": str(uuid.uuid4()),
        "business_constraints": constraints,
        "market_report": mkt,
        "strategy_report": strat,
        "financial_report": fin,
        "ethics_report": eth,
        "validation_report": val,
        "approval_status": "pending",
        "orchestrator_feedback": orchestrator_feedback,
        "confidence_score": 0,
        "final_verdict": None,
        "final_recommendations": [],
        "messages": [],
        "iteration": iteration,
        "max_iterations": 3,
        "user_feedback": None,
        "final_result": None,
    }
    return state

print("_agent_results store + make_fresh_state: OK")

_agent_results store + make_fresh_state: OK


---
## 4. Market Scout Agent

Output yang diharapkan: `MarketScoutReport` dengan fields integer terukur.

> Memanggil LLM + internet search. Estimasi: 30–90 detik.

In [10]:
# ── INTERCEPT: Lihat raw JSON yang dikembalikan LLM sebelum parsing ──
# Cell ini menjalankan LLM sama persis seperti market_scout_node
# tapi print raw response SEBELUM diparse

import importlib
import nodes.market_scout as _ms
importlib.reload(_ms)  # Paksa reload module agar pakai versi terbaru
print(f"market_scout.py loaded from: {_ms.__file__}")

from functions.agent_utils import extract_json, format_constraints, run_planned_search_loop
from functions.llm import get_llm
from langchain_core.messages import HumanMessage, SystemMessage
from tools.internet_search import internet_search

_bc = constraints
_context = [
    "=== BUSINESS CONSTRAINTS ===",
    format_constraints(_bc),
    "\nEvaluasi kelayakan pasar. Return ONLY structured JSON sesuai format."
]

_llm = get_llm(temperature=0.3)
_llm_with_tools = _llm.bind_tools([internet_search])

print("Memanggil LLM + search (intercept mode)...")
_new_msgs, _final = run_planned_search_loop(
    llm=_llm,
    llm_with_tools=_llm_with_tools,
    messages=[
        SystemMessage(content=_ms._SYSTEM_PROMPT),
        HumanMessage(content="\n".join(_context)),
    ],
    tools=[internet_search],
    planning_topics=_ms._SEARCH_TOPICS,
    max_followup_rounds=1,
    agent_name="market_scout_intercept",
    max_search_calls=4,
)

print("\n" + "=" * 60)
print("RAW LLM RESPONSE (sebelum extract_json):")
print("=" * 60)
print(_final.content)
print("=" * 60)

_parsed = extract_json(_final.content)
print("\nPARSED KEYS:", list(_parsed.keys()))
print("\nPARSED VALUES:")
for k, v in _parsed.items():
    print(f"  {k!r}: {str(v)[:120]!r}")


market_scout.py loaded from: d:\Gabriel\Gabriel\Telkom University\Kuliah\Semester VI\Capstone\Multi-Agent-Decision-Support-System\nodes\market_scout.py
Memanggil LLM + search (intercept mode)...


2026-05-27 15:45:12.267  DEBUG  [clario.market_scout_intercept]  [PLAN] Merencanakan 3 query pencarian...
[15:45:12] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-36f3bd38-84d2-437c-b84f-64eaa3f5335b', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are the Market Scout Agent in a multi-agent business planning system.\nYour job is to evaluate market viability and return STRUCTURED data — not essays.\n\nCRITICAL: Output ONLY valid JSON. No prose, no markdown fences, no preamble.\n\nYour analysis covers:\n1. Demand & competition scoring (integer 1-10, not words)\n2. Top 3-5 customer pain points (from real competitor reviews)\n3. Max 3 validated opportunities (specific, not generic)\n4. TAM-SAM-SOM with explicit methodology (population × spending estimate)\n5. Max 3 competitor insights with concrete weaknesses\n\nSCORI


RAW LLM RESPONSE (sebelum extract_json):


{
  "demand_score": 8,
  "competition_level": "high",
  "opportunity_score": 6,
  "market_trend_summary": "Tren WFC (Work From Cafe) di Bandung sangat kuat dengan permintaan tinggi dari mahasiswa dan remote worker untuk ruang kerja nyaman dengan WiFi stabil. Namun pasar sudah jenuh dengan banyak pemain established.",
  "customer_pain_points": [
    "WiFi lambat atau tidak stabil saat jam sibuk, mengganggu produktivitas kerja/tugas",
    "Kursi tidak ergonomis untuk duduk seharian, menyebabkan tidak nyaman saat work session panjang",
    "Harga relatif mahal untuk mahasiswa (Rp30.000-100.000 per item/hari) dengan minimal pembelian",
    "Kurang tersedia colokan listrik di setiap meja, rebutan outlet"
  ],
  "competitors": [
    {
      "name": "Two Cents Coffee",
      "strengths": ["Lokasi strategis", "Brand kuat", "Menu variatif"],
      "weaknesses": ["Harga premium tidak terjangkau mahasiswa", "Sering penuh dan berisik"],
      "market_pos

In [20]:
from nodes.market_scout import market_scout_node

state_ms = make_fresh_state(iteration=0)

print("Menjalankan Market Scout Agent...")
result_ms = market_scout_node(state_ms)

report_ms: MarketScoutReport = result_ms.get("market_report")

# ── Debug print sebelum assertions (mudah diagnosa jika gagal) ──
print("\n=== DEBUG: Raw field values ===")
print(f"  demand_score        : {report_ms.demand_score!r}")
print(f"  competition_level   : {report_ms.competition_level!r}")
print(f"  opportunity_score   : {report_ms.opportunity_score!r}")
print(f"  customer_pain_points: {report_ms.customer_pain_points!r}")
print(f"  validated_opps      : {report_ms.validated_opportunities!r}")
print(f"  confidence_score    : {report_ms.confidence_score!r}")
print(f"  market_sizing_meth  : {report_ms.market_sizing_methodology!r}")
print()

# --- Assertions: verifikasi semua field structured terisi ---
assert report_ms is not None, "market_report seharusnya tidak None!"
assert isinstance(report_ms.demand_score, int), "demand_score harus int"
assert 1 <= report_ms.demand_score <= 10, f"demand_score di luar range 1-10: {report_ms.demand_score}"
assert report_ms.competition_level in ("low", "medium", "high"), f"competition_level invalid: {report_ms.competition_level}"
assert isinstance(report_ms.opportunity_score, int), "opportunity_score harus int"
assert isinstance(report_ms.customer_pain_points, list), "customer_pain_points harus list"
assert len(report_ms.customer_pain_points) > 0, (
    f"customer_pain_points kosong!\n"
    f"  Raw parsed dump:\n  {report_ms!r}"
)
assert isinstance(report_ms.validated_opportunities, list), "validated_opportunities harus list"
assert len(report_ms.validated_opportunities) > 0, (
    f"validated_opportunities kosong!\n"
    f"  Raw dump: {report_ms.validated_opportunities!r}"
)
assert isinstance(report_ms.confidence_score, int), "confidence_score harus int"
assert 1 <= report_ms.confidence_score <= 10, f"confidence_score di luar range: {report_ms.confidence_score}"
assert report_ms.market_sizing_methodology, "market_sizing_methodology tidak boleh kosong"

print("[Market Scout] LULUS ✓")
print(f"  demand_score        : {report_ms.demand_score}/10")
print(f"  competition_level   : {report_ms.competition_level}")
print(f"  opportunity_score   : {report_ms.opportunity_score}/10")
print(f"  confidence_score    : {report_ms.confidence_score}/10")
print(f"  market_trend        : {report_ms.market_trend_summary}")
print(f"  pain_points ({len(report_ms.customer_pain_points)}):")
for pp in report_ms.customer_pain_points:
    print(f"    - {pp}")
print(f"  competitors ({len(report_ms.competitors)}):")
for c in report_ms.competitors:
    print(f"    - {c.name}: weaknesses={c.weaknesses}")
print(f"  TAM: {report_ms.tam_estimate}")
print(f"  SAM: {report_ms.sam_estimate}")
print(f"  SOM: {report_ms.som_estimate}")
print(f"  Methodology: {report_ms.market_sizing_methodology}")
print(f"  Opportunities ({len(report_ms.validated_opportunities)}):")
for op in report_ms.validated_opportunities:
    print(f"    - {op}")
print(f"  Sources: {len(report_ms.source_links)} link(s)")

# Simpan hasil untuk agent berikutnya
_agent_results["report_ms"] = report_ms
print(f"\n✓ Hasil disimpan ke _agent_results['report_ms']")

2026-05-27 15:54:27.874  DEBUG  [clario.market_scout]  ============================================================
2026-05-27 15:54:27.875  DEBUG  [clario.market_scout]  → Market Scout Agent dimulai
2026-05-27 15:54:27.883  DEBUG  [clario.market_scout]  [PLAN] Merencanakan 3 query pencarian...
[15:54:27] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-964514a1-d6ff-4d52-ba99-3d310da9ea88', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are the Market Scout Agent in a multi-agent business planning system.\nYour job is to evaluate market viability and return STRUCTURED data — not essays.\n\nCRITICAL: Output ONLY valid JSON. No prose, no markdown fences, no preamble.\n\nYour analysis covers:\n1. Demand & competition scoring (integer 1-10, not words)\n2. Top 3-5 customer pain points (from real competitor reviews)\n3. Ma

Menjalankan Market Scout Agent...


[15:54:28] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000028F3FAF76E0>
[15:54:28] send_request_headers.started request=<Request [b'POST']>
[15:54:28] send_request_headers.complete
[15:54:28] send_request_body.started request=<Request [b'POST']>
[15:54:28] send_request_body.complete
[15:54:28] receive_response_headers.started request=<Request [b'POST']>
[15:55:11] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 27 May 2026 08:55:12 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'17567'), (b'Connection', b'keep-alive'), (b'server', b'uvicorn'), (b'x-robots-tag', b'noindex'), (b'x-request-id', b'RtAKtQNBJzLdBSPC4NLYR8tZ')])
[15:55:11] HTTP Request: POST https://api.deepinfra.com/v1/openai/chat/completions "HTTP/1.1 200 OK"
[15:55:11] receive_response_body.started request=<Request [b'POST']>
[15:55:11] receive_response_body.complete
[15:55:11] response_closed.started
[15:55:11] response_cl


=== DEBUG: Raw field values ===
  demand_score        : 8
  competition_level   : 'medium'
  opportunity_score   : 7
  customer_pain_points: ['WiFi tidak stabil atau lambat saat jam sibuk mengganggu produktivitas kerja', 'Jumlah colokan listrik tidak memadai, harus berebut dengan pengunjung lain', 'Harga minuman terlalu mahal untuk mahasiswa (di atas Rp 35.000) dengan durasi duduk terbatas']
  validated_opps      : ['Paket berlangganan harian/mingguan untuk mahasiswa dengan harga Rp 25.000-35.000 termasuk unlimited refill kopi lokal', 'Zona quiet work dengan WiFi dedicated 100 Mbps dan colokan di setiap meja (gap dari kompetitor yang WiFi-nya shared)', 'Extended hours hingga 1 AM weekdays dan 24 jam weekend untuk menangkap segmen night workers yang tidak terlayani kompetitor']
  confidence_score    : 7
  market_sizing_meth  : 'Populasi target: 400.000 orang (mahasiswa + pekerja 18-35 di Bandung) × rata-rata spending Rp 500.000/bulan untuk kopi & coworking = TAM. SAM: 20% dari TAM yang

---
## 5. Strategic Architect Agent

Output yang diharapkan: `StrategicReport` dengan SWOT List[str], roadmap, lean canvas.

> Menggunakan hasil Market Scout di atas. Estimasi: 30–90 detik.

In [21]:
from nodes.strategic_architect import strategic_architect_node

state_sa = make_fresh_state(iteration=0, with_market=True)

print("Menjalankan Strategic Architect Agent...")
result_sa = strategic_architect_node(state_sa)

report_sa: StrategicReport = result_sa.get("strategy_report")

# --- Assertions ---
assert report_sa is not None, "strategy_report seharusnya tidak None!"
assert isinstance(report_sa.strengths, list), "strengths harus list"
assert len(report_sa.strengths) > 0, "strengths tidak boleh kosong"
assert isinstance(report_sa.weaknesses, list), "weaknesses harus list"
assert isinstance(report_sa.opportunities, list), "opportunities harus list"
assert isinstance(report_sa.threats, list), "threats harus list"
assert isinstance(report_sa.pestel_highlights, list), "pestel_highlights harus list"
assert len(report_sa.pestel_highlights) > 0, "pestel_highlights tidak boleh kosong"
assert isinstance(report_sa.top_problem_priorities, list), "top_problem_priorities harus list"
assert report_sa.strategic_order_of_operations, "strategic_order_of_operations tidak boleh kosong"
assert "month_1" in report_sa.roadmap, "roadmap harus punya month_1"
assert "month_2" in report_sa.roadmap, "roadmap harus punya month_2"
assert "month_3" in report_sa.roadmap, "roadmap harus punya month_3"
assert isinstance(report_sa.roadmap["month_1"], MonthlyMilestone), "month_1 harus MonthlyMilestone"
assert report_sa.roadmap["month_1"].objective, "month_1 objective tidak boleh kosong"
assert len(report_sa.roadmap["month_1"].kpis) > 0, "month_1 KPIs tidak boleh kosong"
assert report_sa.value_proposition, "value_proposition tidak boleh kosong"
assert 1 <= report_sa.confidence_score <= 10, f"confidence_score di luar range: {report_sa.confidence_score}"

print("\n[Strategic Architect] LULUS ✓")
print(f"  confidence_score  : {report_sa.confidence_score}/10")
print(f"  strengths ({len(report_sa.strengths)}):")
for s in report_sa.strengths:
    print(f"    + {s}")
print(f"  weaknesses ({len(report_sa.weaknesses)}):")
for w in report_sa.weaknesses:
    print(f"    - {w}")
print(f"  opportunities ({len(report_sa.opportunities)}):")
for o in report_sa.opportunities:
    print(f"    → {o}")
print(f"  threats ({len(report_sa.threats)}):")
for t in report_sa.threats:
    print(f"    ! {t}")
print(f"  pestel_highlights ({len(report_sa.pestel_highlights)}):")
for p in report_sa.pestel_highlights:
    print(f"    · {p}")
print(f"  top_problem_priorities: {report_sa.top_problem_priorities}")
print(f"  order_of_ops: {report_sa.strategic_order_of_operations}")
print(f"  Roadmap:")
for month, ms in report_sa.roadmap.items():
    print(f"    {month}: {ms.objective} | KPIs: {ms.kpis}")
print(f"  UVP: {report_sa.value_proposition}")
print(f"  Segments: {report_sa.customer_segments}")
print(f"  Channels: {report_sa.channels}")
print(f"  Revenue: {report_sa.revenue_streams}")

# Simpan hasil untuk agent berikutnya
_agent_results["report_sa"] = report_sa
print(f"\n✓ Hasil disimpan ke _agent_results['report_sa']")

2026-05-27 16:01:21.046  DEBUG  [clario.strategic_architect]  ============================================================
2026-05-27 16:01:21.051  DEBUG  [clario.strategic_architect]  → Strategic Architect Agent dimulai
2026-05-27 16:01:21.086  DEBUG  [clario.strategic_architect]  [PLAN] Merencanakan 2 query pencarian...
[16:01:21] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-0c9e91c0-e867-42dc-868a-74d151ec3a71', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are the Strategic Architect Agent in a multi-agent business planning system.\nYour job is to build a data-backed strategic execution model and return STRUCTURED data — not essays.\n\nCRITICAL: Output ONLY valid JSON. No prose, no markdown fences, no preamble.\n\nRULES:\n- SWOT: MAXIMUM 3 bullet points per section. Be specific, not generic.\n- PESTEL: MAXIMU

Menjalankan Strategic Architect Agent...


[16:01:21] connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000028F4188C950>
[16:01:21] start_tls.started ssl_context=<ssl.SSLContext object at 0x0000028F3DD55250> server_hostname='api.deepinfra.com' timeout=None
[16:01:21] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000028F411CAAB0>
[16:01:21] send_request_headers.started request=<Request [b'POST']>
[16:01:21] send_request_headers.complete
[16:01:21] send_request_body.started request=<Request [b'POST']>
[16:01:21] send_request_body.complete
[16:01:21] receive_response_headers.started request=<Request [b'POST']>
[16:02:50] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 27 May 2026 09:02:51 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'17278'), (b'Connection', b'keep-alive'), (b'server', b'uvicorn'), (b'x-robots-tag', b'noindex'), (b'x-request-id', b'RMEMCzt1NQC0KKu5O9eF4Su0')])
[16:02:50] HTTP Reques


[Strategic Architect] LULUS ✓
  confidence_score  : 7/10
  strengths (3):
    + WiFi dedicated 100 Mbps dengan zona quiet work yang tidak dimiliki 70% kompetitor di Bandung
    + Paket berlangganan Rp 25.000-35.000 unlimited refill 40% lebih murah dari harga rata-rata kopi spesialti Bandung (Rp 45.000)
    + Extended hours hingga 1 AM weekdays dan 24 jam weekend menangkap segmen night workers yang 0% terlayani kompetitor
  weaknesses (3):
    - Budget Rp 50-150 juta terbatas untuk renovasi dan peralatan dibanding chain coffee shop dengan modal Rp 500 juta+
    - Tim beginner tanpa pengalaman operasional F&B berisiko tinggi pada efisiensi biaya dan quality control
    - Lokasi offline tunggal bergantung pada foot traffic lokal tanpa diversifikasi revenue stream
  opportunities (3):
    → Tren Work From Cafe Bandung meningkat 35% YoY dengan 60% mahasiswa dan remote worker mencari alternatif kantor
    → Gap pasar subscription model belum diadopsi kompetitor utama di Bandung (Anomali, Ko

---
## 6. Financial Analyst Agent

Output yang diharapkan: `FinancialAnalysisReport` dengan semua angka sebagai **integer IDR**.

> Estimasi: 30–90 detik.

In [12]:
from nodes.financial_analyst import financial_analyst_node

state_fa = make_fresh_state(iteration=0, with_market=True, with_strategy=True)

print("Menjalankan Financial Analyst Agent...")
result_fa = financial_analyst_node(state_fa)

report_fa: FinancialAnalysisReport = result_fa.get("financial_report")

# --- Assertions ---
assert report_fa is not None, "financial_report seharusnya tidak None!"

# Semua angka harus int IDR
assert isinstance(report_fa.estimated_monthly_revenue_rp, int), "revenue harus int IDR"
assert report_fa.estimated_monthly_revenue_rp > 0, "revenue harus > 0"
assert isinstance(report_fa.estimated_monthly_net_profit_rp, int), "net_profit harus int IDR"
assert isinstance(report_fa.estimated_startup_cost_rp, int), "startup_cost harus int IDR"
assert report_fa.estimated_startup_cost_rp > 0, "startup_cost harus > 0"
assert isinstance(report_fa.bep_units_per_month, int), "bep_units harus int"
assert isinstance(report_fa.bep_revenue_rp, int), "bep_revenue harus int IDR"
assert isinstance(report_fa.estimated_break_even_months, int), "break_even harus int"
assert isinstance(report_fa.cash_runway_months, int), "cash_runway harus int"

# Net profit tidak boleh > revenue (cek matematika dasar)
assert report_fa.estimated_monthly_net_profit_rp <= report_fa.estimated_monthly_revenue_rp, \
    f"Net profit ({report_fa.estimated_monthly_net_profit_rp}) tidak boleh melebihi revenue ({report_fa.estimated_monthly_revenue_rp})"

# Verdict harus valid
assert report_fa.financial_feasibility_verdict in ("LAYAK", "LAYAK_DENGAN_CATATAN", "TIDAK_LAYAK"), \
    f"Verdict invalid: {report_fa.financial_feasibility_verdict}"

# Sensitivity scenarios
assert isinstance(report_fa.sensitivity_scenarios, list), "sensitivity_scenarios harus list"
assert len(report_fa.sensitivity_scenarios) >= 1, "Minimal 1 sensitivity scenario"
for sc in report_fa.sensitivity_scenarios:
    assert isinstance(sc, SensitivityScenario), "Setiap scenario harus SensitivityScenario"
    assert isinstance(sc.net_profit_rp, int), f"net_profit_rp scenario '{sc.label}' harus int"

# Assumptions harus ada
assert isinstance(report_fa.assumptions, list), "assumptions harus list"
assert len(report_fa.assumptions) > 0, "assumptions tidak boleh kosong"

# Confidence score
assert 1 <= report_fa.confidence_score <= 10, f"confidence_score di luar range: {report_fa.confidence_score}"
assert 1 <= report_fa.affordability_score <= 10, f"affordability_score di luar range: {report_fa.affordability_score}"

print("\n[Financial Analyst] LULUS ✓")
print(f"  verdict             : {report_fa.financial_feasibility_verdict}")
print(f"  confidence_score    : {report_fa.confidence_score}/10")
print(f"  affordability_score : {report_fa.affordability_score}/10")
print()
print("  PROYEKSI BULANAN:")
print(f"    Revenue  : Rp {report_fa.estimated_monthly_revenue_rp:>15,}")
print(f"    COGS     : Rp {report_fa.estimated_monthly_cogs_rp:>15,}")
print(f"    Fixed    : Rp {report_fa.estimated_monthly_fixed_costs_rp:>15,}")
print(f"    Net Pft  : Rp {report_fa.estimated_monthly_net_profit_rp:>15,}")
print()
print("  METRIK KEPUTUSAN:")
print(f"    Startup Cost  : Rp {report_fa.estimated_startup_cost_rp:,}")
print(f"    BEP Units     : {report_fa.bep_units_per_month} unit/bulan")
print(f"    BEP Revenue   : Rp {report_fa.bep_revenue_rp:,}")
print(f"    Break-Even    : {report_fa.estimated_break_even_months} bulan")
print(f"    Cash Runway   : {report_fa.cash_runway_months} bulan")
print()
print(f"  SENSITIVITY ({len(report_fa.sensitivity_scenarios)} skenario):")
for sc in report_fa.sensitivity_scenarios:
    print(f"    {sc.label}: Rp {sc.net_profit_rp:,}")
print()
print(f"  ASSUMPTIONS ({len(report_fa.assumptions)}):")
for a in report_fa.assumptions:
    print(f"    · {a}")
print()
print(f"  KEY RISKS ({len(report_fa.key_financial_risks)}):")
for r in report_fa.key_financial_risks:
    print(f"    ⚠ {r}")

# Simpan hasil untuk agent berikutnya
_agent_results["report_fa"] = report_fa
print(f"\n✓ Hasil disimpan ke _agent_results['report_fa']")

2026-05-27 15:48:24.995  DEBUG  [clario.financial_analyst]  ============================================================
2026-05-27 15:48:24.998  DEBUG  [clario.financial_analyst]  → Financial Analyst Agent dimulai
2026-05-27 15:48:25.005  DEBUG  [clario.financial_analyst]  [PLAN] Merencanakan 2 query pencarian...
[15:48:25] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-44ff80df-35d0-4b9d-aa25-e111259b47dd', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are the Financial Analyst Agent in a multi-agent business planning system.\nYour job is to calculate financial projections and return STRUCTURED numerical data — not essays.\n\nCRITICAL: Output ONLY valid JSON. All monetary values MUST be integers in Indonesian Rupiah (IDR). No "Rp" prefix in number fields — just the integer.\n\nCALCULATION REQUIREMENTS:\n1. Monthl

Menjalankan Financial Analyst Agent...


[15:48:50] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 27 May 2026 08:48:50 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'9630'), (b'Connection', b'keep-alive'), (b'server', b'uvicorn'), (b'x-robots-tag', b'noindex'), (b'x-request-id', b'RA2ETpz0gg3kfZwRLrJq317E')])
[15:48:50] HTTP Request: POST https://api.deepinfra.com/v1/openai/chat/completions "HTTP/1.1 200 OK"
[15:48:50] receive_response_body.started request=<Request [b'POST']>
[15:48:50] receive_response_body.complete
[15:48:50] response_closed.started
[15:48:50] response_closed.complete
[15:48:50] HTTP Response: POST https://api.deepinfra.com/v1/openai/chat/completions "200 OK" Headers({'date': 'Wed, 27 May 2026 08:48:50 GMT', 'content-type': 'application/json', 'content-length': '9630', 'connection': 'keep-alive', 'server': 'uvicorn', 'x-robots-tag': 'noindex', 'x-request-id': 'RA2ETpz0gg3kfZwRLrJq317E'})
[15:48:50] request_id: RA2ETpz0gg3kfZwRLrJq317E
2026-0


[Financial Analyst] LULUS ✓
  verdict             : LAYAK
  confidence_score    : 7/10
  affordability_score : 8/10

  PROYEKSI BULANAN:
    Revenue  : Rp      65,000,000
    COGS     : Rp      18,200,000
    Fixed    : Rp      28,500,000
    Net Pft  : Rp      18,300,000

  METRIK KEPUTUSAN:
    Startup Cost  : Rp 145,000,000
    BEP Units     : 974 unit/bulan
    BEP Revenue   : Rp 43,830,000
    Break-Even    : 8 bulan
    Cash Runway   : 0 bulan

  SENSITIVITY (3 skenario):
    Volume penjualan turun 20%: Rp 8,600,000
    COGS naik 15%: Rp 15,470,000
    Marketing cost naik 50%: Rp 17,050,000

  ASSUMPTIONS (6):
    · Diasumsikan harga rata-rata transaksi F&B Rp 45.000 per pelanggan
    · Diasumsikan 40-45 pelanggan F&B per hari dan 10 pengguna coworking per hari
    · Diasumsikan sewa lokasi Rp 8.000.000/bulan berdasarkan benchmark area Bandung untuk ruko 80-100m²
    · Diasumsikan COGS 35% dari revenue F&B saja (coworking dan event memiliki margin lebih tinggi)
    · Diasumsikan

---
## 7. Ethics Agent

Output yang diharapkan: `EthicsAnalysisReport` dengan lists perizinan dan checklist structured.

> Estimasi: 20–60 detik.

In [13]:
from nodes.ethics_agent import ethics_agent_node

state_ea = make_fresh_state(iteration=0, with_market=True, with_strategy=True)

print("Menjalankan Ethics Agent...")
result_ea = ethics_agent_node(state_ea)

report_ea: EthicsAnalysisReport = result_ea.get("ethics_report")

# --- Assertions ---
assert report_ea is not None, "ethics_report seharusnya tidak None!"
assert isinstance(report_ea.mandatory_permits, list), "mandatory_permits harus list"
assert len(report_ea.mandatory_permits) > 0, "mandatory_permits tidak boleh kosong"
assert isinstance(report_ea.sector_specific_permits, list), "sector_specific_permits harus list"
assert isinstance(report_ea.compliance_checklist_perizinan, list), "checklist_perizinan harus list"
assert isinstance(report_ea.compliance_checklist_pajak, list), "checklist_pajak harus list"
assert isinstance(report_ea.compliance_checklist_legalitas, list), "checklist_legalitas harus list"
assert isinstance(report_ea.critical_risk_alerts, list), "critical_risk_alerts harus list"
assert report_ea.legal_risk_level in ("low", "medium", "high"), \
    f"legal_risk_level invalid: {report_ea.legal_risk_level}"
assert report_ea.bureaucracy_complexity in ("RENDAH", "SEDANG", "TINGGI"), \
    f"bureaucracy_complexity invalid: {report_ea.bureaucracy_complexity}"
assert report_ea.bureaucracy_estimated_duration, "estimated_duration tidak boleh kosong"
assert 1 <= report_ea.compliance_score <= 10, f"compliance_score di luar range: {report_ea.compliance_score}"
assert 1 <= report_ea.confidence_score <= 10, f"confidence_score di luar range: {report_ea.confidence_score}"

print("\n[Ethics Agent] LULUS ✓")
print(f"  legal_risk_level    : {report_ea.legal_risk_level}")
print(f"  compliance_score    : {report_ea.compliance_score}/10")
print(f"  confidence_score    : {report_ea.confidence_score}/10")
print(f"  complexity          : {report_ea.bureaucracy_complexity}")
print(f"  duration            : {report_ea.bureaucracy_estimated_duration}")
print()
print(f"  IZIN WAJIB ({len(report_ea.mandatory_permits)}):")
for p in report_ea.mandatory_permits:
    print(f"    ✓ {p}")
print(f"  IZIN SEKTORAL ({len(report_ea.sector_specific_permits)}):")
for p in report_ea.sector_specific_permits:
    print(f"    ✓ {p}")
print(f"  CHECKLIST PERIZINAN:")
for c in report_ea.compliance_checklist_perizinan:
    print(f"    □ {c}")
print(f"  CHECKLIST PAJAK:")
for c in report_ea.compliance_checklist_pajak:
    print(f"    □ {c}")
print(f"  CHECKLIST LEGALITAS:")
for c in report_ea.compliance_checklist_legalitas:
    print(f"    □ {c}")
print(f"  RISK ALERTS ({len(report_ea.critical_risk_alerts)}):")
for r in report_ea.critical_risk_alerts:
    print(f"    🔴 {r}")

# Simpan hasil untuk agent berikutnya
_agent_results["report_ea"] = report_ea
print(f"\n✓ Hasil disimpan ke _agent_results['report_ea']")

2026-05-27 15:49:31.809  DEBUG  [clario.ethics_agent]  ============================================================
2026-05-27 15:49:31.810  DEBUG  [clario.ethics_agent]  → Ethics & Compliance Agent dimulai
2026-05-27 15:49:31.818  DEBUG  [clario.ethics_agent]  [PLAN] Merencanakan 2 query pencarian...
[15:49:31] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-ee58e7e5-a740-45d4-ba82-e9f5695e1e4f', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are the Ethics & Compliance Agent in a multi-agent business planning system.\nYour job is to evaluate Indonesian regulatory compliance and return STRUCTURED data — not essays.\n\nCRITICAL: Output ONLY valid JSON. No prose, no markdown fences, no preamble.\n\nRULES-BASED LICENSING LOGIC (apply strictly):\n- F&B / Kuliner: → NIB (OSS), PIRT (produksi rumahan) atau BPOM (skala pab

Menjalankan Ethics Agent...


[15:50:12] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 27 May 2026 08:50:11 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'14496'), (b'Connection', b'keep-alive'), (b'server', b'uvicorn'), (b'x-robots-tag', b'noindex'), (b'x-request-id', b'RBNxSlT7tRDnPxiaHfkNpzaf')])
[15:50:12] HTTP Request: POST https://api.deepinfra.com/v1/openai/chat/completions "HTTP/1.1 200 OK"
[15:50:12] receive_response_body.started request=<Request [b'POST']>
[15:50:12] receive_response_body.complete
[15:50:12] response_closed.started
[15:50:12] response_closed.complete
[15:50:12] HTTP Response: POST https://api.deepinfra.com/v1/openai/chat/completions "200 OK" Headers({'date': 'Wed, 27 May 2026 08:50:11 GMT', 'content-type': 'application/json', 'content-length': '14496', 'connection': 'keep-alive', 'server': 'uvicorn', 'x-robots-tag': 'noindex', 'x-request-id': 'RBNxSlT7tRDnPxiaHfkNpzaf'})
[15:50:12] request_id: RBNxSlT7tRDnPxiaHfkNpzaf
2026


[Ethics Agent] LULUS ✓
  legal_risk_level    : medium
  compliance_score    : 7/10
  confidence_score    : 8/10
  complexity          : SEDANG
  duration            : 2–4 minggu untuk NIB + Sertifikat Standar; 4–8 minggu untuk Sertifikasi Halal; 1–2 minggu untuk PIRT

  IZIN WAJIB (3):
    ✓ NIB (Nomor Induk Berusaha) via OSS dengan KBLI 56303 (Aktivitas Rumah Minum/Kafe)
    ✓ NPWP Badan/Perorangan di KPP terdekat
    ✓ Sertifikat Standar (jika kapasitas tempat duduk ≥50 kursi)
  IZIN SEKTORAL (4):
    ✓ Sertifikasi Halal BPJPH (wajib untuk produk makanan/minuman yang menargetkan konsumen Muslim)
    ✓ PIRT dari Dinas Kesehatan Kota Bandung (jika produksi makanan rumahan skala kecil)
    ✓ Izin Lokasi/Tata Ruang dari Pemda Kota Bandung (kesesuaian RTRW)
    ✓ SPPL (Surat Pernyataan Kesanggupan Pengelolaan Lingkungan) untuk usaha skala kecil-menengah
  CHECKLIST PERIZINAN:
    □ Daftar di OSS (oss.go.id) untuk mendapatkan NIB dengan KBLI 56303
    □ Ajukan kesesuaian tata ruang ke Din

---
## 8. Validation Layer

Tidak memanggil LLM — murni rule-based. Harus selesai < 1 detik.

Test ini mencakup:
- Normal case (semua angka konsisten)
- Contradiction case (angka ngawur untuk trigger rules)

In [14]:
from nodes.validation_layer import validation_layer_node
import time

# ─── Test 1: Normal case — pakai hasil agent nyata ───
state_vl = make_fresh_state(
    iteration=0,
    with_market=True,
    with_strategy=True,
    with_financial=True,
    with_ethics=True,
)

t0 = time.perf_counter()
result_vl = validation_layer_node(state_vl)
elapsed = time.perf_counter() - t0

report_vl: ValidationReport = result_vl.get("validation_report")

assert report_vl is not None, "validation_report seharusnya tidak None!"
assert isinstance(report_vl.contradictions, list), "contradictions harus list"
assert isinstance(report_vl.missing_assumptions, list), "missing_assumptions harus list"
assert isinstance(report_vl.unrealistic_claims, list), "unrealistic_claims harus list"
assert isinstance(report_vl.auto_flags, list), "auto_flags harus list"
assert isinstance(report_vl.is_valid, bool), "is_valid harus bool"
assert 1 <= report_vl.overall_confidence <= 10, f"overall_confidence di luar range: {report_vl.overall_confidence}"
assert elapsed < 2.0, f"Validation layer terlalu lambat ({elapsed:.2f}s) — seharusnya < 2 detik (no LLM)"

print("[Validation Layer — Normal Case] LULUS ✓")
print(f"  Waktu eksekusi    : {elapsed:.3f}s (no LLM call)")
print(f"  is_valid          : {report_vl.is_valid}")
print(f"  overall_confidence: {report_vl.overall_confidence}/10")
print(f"  contradictions    : {len(report_vl.contradictions)}")
for c in report_vl.contradictions:
    print(f"    [{c.severity.upper()}] {c.description}")
print(f"  missing_assumptions: {report_vl.missing_assumptions}")
print(f"  unrealistic_claims: {report_vl.unrealistic_claims}")
print(f"  auto_flags: {report_vl.auto_flags}")

# Simpan hasil untuk agent berikutnya
_agent_results["report_vl"] = report_vl
print(f"\n✓ Hasil disimpan ke _agent_results['report_vl']")

2026-05-27 15:50:44.322  DEBUG  [clario.validation_layer]  ============================================================
2026-05-27 15:50:44.324  DEBUG  [clario.validation_layer]  → Validation Layer dimulai
2026-05-27 15:50:44.326  DEBUG  [clario.validation_layer]     Validation selesai — Critical: 0 | Warning: 0 | Missing: 0 | Confidence: 7/10
2026-05-27 15:50:44.327  DEBUG  [clario.validation_layer]  ✓ Validation Layer selesai dalam 0.00s
2026-05-27 15:50:44.328  DEBUG  [clario.validation_layer]  ============================================================


[Validation Layer — Normal Case] LULUS ✓
  Waktu eksekusi    : 0.006s (no LLM call)
  is_valid          : True
  overall_confidence: 7/10
  contradictions    : 0
  missing_assumptions: []
  unrealistic_claims: []
  auto_flags: []

✓ Hasil disimpan ke _agent_results['report_vl']


In [15]:
# ─── Test 2: Contradiction case — angka sengaja dibuat ngawur ───
# Skenario: demand_score rendah (2) tapi revenue proyeksi Rp 500 juta
# + competition HIGH tapi margin 90%
# + startup cost jauh melebihi budget
# + break_even > cash_runway

bad_market = MarketScoutReport(
    demand_score=2,                 # SANGAT RENDAH
    competition_level="high",       # TINGGI
    opportunity_score=3,
    market_trend_summary="Pasar sangat jenuh.",
    customer_pain_points=["Tidak ada pain point"],
    competitors=[],
    tam_estimate="Rp 1 Triliun",
    sam_estimate="Rp 100 Miliar",
    som_estimate="Rp 5 Miliar",
    market_sizing_methodology="Estimasi kasar",
    validated_opportunities=[],
    source_links=[],
    confidence_score=3,
)

bad_financial = FinancialAnalysisReport(
    estimated_monthly_revenue_rp=500_000_000,   # Rp 500 juta/bulan — TIDAK REALISTIS dengan demand rendah
    estimated_monthly_cogs_rp=25_000_000,        # COGS sangat kecil → margin 95% → TIDAK WAJAR dengan competition HIGH
    estimated_monthly_fixed_costs_rp=20_000_000,
    estimated_monthly_net_profit_rp=455_000_000,
    bep_units_per_month=100,
    bep_revenue_rp=20_000_000,
    cash_runway_months=3,                        # Cash runway 3 bulan
    estimated_startup_cost_rp=500_000_000,       # Startup cost Rp 500 juta >> budget max Rp 150 juta
    estimated_break_even_months=18,              # Break-even 18 bulan >> cash runway 3 bulan
    sensitivity_scenarios=[],
    industry_benchmark_notes="",
    financial_feasibility_verdict="LAYAK",        # Verdict 'LAYAK' padahal ngawur
    key_financial_risks=[],
    assumptions=[],                              # Tidak ada assumptions
    affordability_score=3,
    confidence_score=3,
)

bad_ethics = EthicsAnalysisReport(
    mandatory_permits=["NIB"],
    sector_specific_permits=[],
    compliance_checklist_perizinan=[],
    compliance_checklist_pajak=[],
    compliance_checklist_legalitas=[],
    critical_risk_alerts=["Risiko tinggi pelanggaran regulasi pangan"],
    legal_risk_level="high",                     # Legal risk HIGH
    bureaucracy_estimated_duration="6 bulan",
    bureaucracy_complexity="TINGGI",
    compliance_score=3,
    confidence_score=3,
)

bad_state: EBPState = {
    "state_id": str(uuid.uuid4()),
    "business_constraints": constraints,
    "market_report": bad_market,
    "strategy_report": None,
    "financial_report": bad_financial,
    "ethics_report": bad_ethics,
    "validation_report": None,
    "approval_status": "pending",
    "orchestrator_feedback": None,
    "confidence_score": 0,
    "final_verdict": None,
    "final_recommendations": [],
    "messages": [],
    "iteration": 0,
    "max_iterations": 3,
    "user_feedback": None,
    "final_result": None,
}

result_bad = validation_layer_node(bad_state)
report_bad: ValidationReport = result_bad.get("validation_report")

# Harus ada contradiction flags
assert len(report_bad.contradictions) > 0, "Harus ada contradiction flags untuk angka ngawur ini!"

# Harus ada critical flag (demand rendah + revenue tinggi)
has_critical = any(c.severity == "critical" for c in report_bad.contradictions)
assert has_critical, "Harus ada minimal 1 critical contradiction!"

# is_valid harus False karena ada critical
assert report_bad.is_valid is False, "is_valid harus False karena ada critical contradiction"

# Confidence harus rendah
assert report_bad.overall_confidence < 6, \
    f"Confidence harus rendah untuk kasus buruk, dapat: {report_bad.overall_confidence}"

# Missing assumptions harus terdeteksi
assert len(report_bad.missing_assumptions) > 0, "Missing assumptions harus terdeteksi"

print("[Validation Layer — Contradiction Case] LULUS ✓")
print(f"  is_valid           : {report_bad.is_valid} (benar — ada critical contradiction)")
print(f"  overall_confidence : {report_bad.overall_confidence}/10")
print(f"  total contradictions: {len(report_bad.contradictions)}")
print()
for c in report_bad.contradictions:
    print(f"  [{c.severity.upper()}] {c.description}")
print()
print(f"  missing_assumptions:")
for m in report_bad.missing_assumptions:
    print(f"    · {m}")
print(f"  unrealistic_claims:")
for u in report_bad.unrealistic_claims:
    print(f"    ! {u}")

2026-05-27 15:50:44.353  DEBUG  [clario.validation_layer]  ============================================================
2026-05-27 15:50:44.355  DEBUG  [clario.validation_layer]  → Validation Layer dimulai
2026-05-27 15:50:44.356  DEBUG  [clario.validation_layer]     Validation selesai — Critical: 4 | Warning: 0 | Missing: 1 | Confidence: 1/10
2026-05-27 15:50:44.357  DEBUG  [clario.validation_layer]  ✓ Validation Layer selesai dalam 0.00s
2026-05-27 15:50:44.358  DEBUG  [clario.validation_layer]  ============================================================


[Validation Layer — Contradiction Case] LULUS ✓
  is_valid           : False (benar — ada critical contradiction)
  overall_confidence : 1/10
  total contradictions: 4

  [CRITICAL] Market competition=HIGH namun projected gross margin=1011%. Di pasar kompetitif, margin setinggi ini tidak realistis tanpa differensiasi kuat.
  [CRITICAL] Market demand_score=2/10 (sangat rendah) namun projected monthly revenue=Rp 500,000,000. Proyeksi revenue tidak konsisten dengan sinyal permintaan pasar.
  [CRITICAL] BEP diperkirakan bulan ke-18, namun cash runway hanya 3 bulan. Bisnis akan kehabisan modal sebelum mencapai titik impas.
  [CRITICAL] Estimated startup cost=Rp 500,000,000 melebihi 150% dari budget maksimum user (Rp 150,000,000). Bisnis tidak terjangkau dengan modal yang tersedia.

  missing_assumptions:
    · Financial Analyst tidak mendeklarasikan asumsi kalkulasi. Angka tidak dapat diverifikasi.
  unrealistic_claims:


In [16]:
# ─── Test 3: Edge case — net profit > revenue (matematika tidak mungkin) ───
broken_financial = FinancialAnalysisReport(
    estimated_monthly_revenue_rp=10_000_000,
    estimated_monthly_cogs_rp=1_000_000,
    estimated_monthly_fixed_costs_rp=1_000_000,
    estimated_monthly_net_profit_rp=15_000_000,   # Net profit > revenue — IMPOSSIBLE
    bep_units_per_month=50,
    bep_revenue_rp=5_000_000,
    cash_runway_months=6,
    estimated_startup_cost_rp=50_000_000,
    estimated_break_even_months=5,
    sensitivity_scenarios=[],
    industry_benchmark_notes="",
    financial_feasibility_verdict="LAYAK",
    key_financial_risks=[],
    assumptions=["Asumsi A"],
    affordability_score=7,
    confidence_score=7,
)

edge_state: EBPState = {
    "state_id": str(uuid.uuid4()),
    "business_constraints": constraints,
    "market_report": None,
    "strategy_report": None,
    "financial_report": broken_financial,
    "ethics_report": None,
    "validation_report": None,
    "approval_status": "pending",
    "orchestrator_feedback": None,
    "confidence_score": 0,
    "final_verdict": None,
    "final_recommendations": [],
    "messages": [],
    "iteration": 0,
    "max_iterations": 3,
    "user_feedback": None,
    "final_result": None,
}

result_edge = validation_layer_node(edge_state)
report_edge: ValidationReport = result_edge.get("validation_report")

# Net profit > revenue harus masuk unrealistic_claims
has_impossible_math = any(
    "Net profit" in claim or "revenue" in claim
    for claim in report_edge.unrealistic_claims
)
assert has_impossible_math, f"Net profit > revenue tidak terdeteksi! claims: {report_edge.unrealistic_claims}"

print("[Validation Layer — Edge Case: Net Profit > Revenue] LULUS ✓")
print(f"  unrealistic_claims: {report_edge.unrealistic_claims}")

2026-05-27 15:50:44.379  DEBUG  [clario.validation_layer]  ============================================================
2026-05-27 15:50:44.381  DEBUG  [clario.validation_layer]  → Validation Layer dimulai
2026-05-27 15:50:44.383  DEBUG  [clario.validation_layer]     Validation selesai — Critical: 0 | Warning: 0 | Missing: 0 | Confidence: 6/10
2026-05-27 15:50:44.384  DEBUG  [clario.validation_layer]  ✓ Validation Layer selesai dalam 0.00s
2026-05-27 15:50:44.386  DEBUG  [clario.validation_layer]  ============================================================


[Validation Layer — Edge Case: Net Profit > Revenue] LULUS ✓
  unrealistic_claims: ['Net profit (Rp 15,000,000) > monthly revenue (Rp 10,000,000). Ini secara matematika tidak mungkin — periksa kalkulasi agent.']


---
## 9. Lead Orchestrator

Bukan essay generator — hanya membaca structured state dan output verdict + top 3 recs.

> Estimasi: 10–30 detik (LLM call kecil).

In [17]:
from nodes.lead_orchestrator import lead_orchestrator_node

# ─── Test first pass (tidak ada report) → harus return 'pending' ───
state_first = make_fresh_state(iteration=0)   # tidak ada report
result_first = lead_orchestrator_node(state_first)

assert result_first.get("approval_status") == "pending", \
    f"First pass harus 'pending', dapat: {result_first.get('approval_status')}"
print("[Orchestrator — First Pass] LULUS ✓ (approval_status='pending', no LLM called)")

2026-05-27 15:50:44.420  DEBUG  [clario.lead_orchestrator]  ============================================================
2026-05-27 15:50:44.422  DEBUG  [clario.lead_orchestrator]  → Lead Orchestrator dimulai
2026-05-27 15:50:44.422  DEBUG  [clario.lead_orchestrator]    First pass → routing ke Market Scout


[Orchestrator — First Pass] LULUS ✓ (approval_status='pending', no LLM called)


In [18]:
# ─── Test full evaluation — semua reports + validation tersedia ───
state_orch = make_fresh_state(
    iteration=0,
    with_market=True,
    with_strategy=True,
    with_financial=True,
    with_ethics=True,
    with_validation=True,
)

# Pastikan validation_report tersedia
if state_orch["validation_report"] is None:
    # Jalankan validation dulu jika belum
    vl_result = validation_layer_node(state_orch)
    state_orch["validation_report"] = vl_result["validation_report"]
    state_orch["confidence_score"] = vl_result["confidence_score"]

print("Menjalankan Lead Orchestrator...")
result_orch = lead_orchestrator_node(state_orch)

# --- Assertions ---
status = result_orch.get("approval_status")
assert status in ("approved", "rejected", "pending"), f"approval_status invalid: {status}"

verdict = result_orch.get("final_verdict")
assert verdict in ("PROCEED", "PROCEED_WITH_CAUTION", "PIVOT_RECOMMENDED", "NOT_RECOMMENDED", None), \
    f"final_verdict invalid: {verdict}"

recs = result_orch.get("final_recommendations", [])
assert isinstance(recs, list), "final_recommendations harus list"
assert len(recs) <= 3, f"Max 3 rekomendasi, dapat {len(recs)}"

# Orchestrator tidak boleh generate essay — periksa messages tetap concise
messages = result_orch.get("messages", [])
if messages:
    msg_content = messages[0].content if hasattr(messages[0], 'content') else ""
    assert len(msg_content) < 500, \
        f"Orchestrator message terlalu panjang ({len(msg_content)} chars) — seharusnya concise summary"

print("[Lead Orchestrator — Full Evaluation] LULUS ✓")
print(f"  approval_status      : {status}")
print(f"  final_verdict        : {verdict}")
print(f"  final_recommendations ({len(recs)}):")
for i, rec in enumerate(recs, 1):
    print(f"    {i}. {rec}")
feedback = result_orch.get("orchestrator_feedback", "")
if feedback:
    print(f"  orchestrator_feedback: {feedback[:200]}")
if messages:
    msg_content = messages[0].content if hasattr(messages[0], 'content') else ""
    print(f"  message length       : {len(msg_content)} chars")

# Simpan hasil untuk ringkasan
_agent_results["result_orch"] = result_orch

2026-05-27 15:50:44.446  DEBUG  [clario.lead_orchestrator]  ============================================================
2026-05-27 15:50:44.448  DEBUG  [clario.lead_orchestrator]  → Lead Orchestrator dimulai
2026-05-27 15:50:44.452  DEBUG  [clario.lead_orchestrator]    Evaluating iteration 0...
[15:50:44] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-5ac83ac7-a836-47d4-9946-51cc55aeef8c', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are the Lead Orchestrator of a multi-agent business planning system.\nYou receive STRUCTURED data from 4 specialist agents + a validation report.\nYour job: issue a final verdict and top 3 recommendations. Be concise and decisive.\n\nVERDICT OPTIONS:\n- "PROCEED": semua sinyal positif, risiko terkendali\n- "PROCEED_WITH_CAUTION": ada risiko signifikan tapi bisnis viable, perlu mitiga

Menjalankan Lead Orchestrator...


[15:51:13] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 27 May 2026 08:51:13 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'9628'), (b'Connection', b'keep-alive'), (b'server', b'uvicorn'), (b'x-robots-tag', b'noindex'), (b'x-request-id', b'RPJvABTUTCYPTtwUcxDHdae9')])
[15:51:13] HTTP Request: POST https://api.deepinfra.com/v1/openai/chat/completions "HTTP/1.1 200 OK"
[15:51:13] receive_response_body.started request=<Request [b'POST']>
[15:51:13] receive_response_body.complete
[15:51:13] response_closed.started
[15:51:13] response_closed.complete
[15:51:13] HTTP Response: POST https://api.deepinfra.com/v1/openai/chat/completions "200 OK" Headers({'date': 'Wed, 27 May 2026 08:51:13 GMT', 'content-type': 'application/json', 'content-length': '9628', 'connection': 'keep-alive', 'server': 'uvicorn', 'x-robots-tag': 'noindex', 'x-request-id': 'RPJvABTUTCYPTtwUcxDHdae9'})
[15:51:13] request_id: RPJvABTUTCYPTtwUcxDHdae9
2026-0

[Lead Orchestrator — Full Evaluation] LULUS ✓
  approval_status      : approved
  final_verdict        : PROCEED_WITH_CAUTION
  final_recommendations (3):
    1. Segera urus NIB dan Sertifikasi Halal untuk mitigasi risiko hukum dan denda
    2. Secure lokasi strategis dengan foot traffic tinggi sebelum anggaran habis
    3. Siapkan dana cadangan operasional karena cash runway saat ini 0 bulan
  message length       : 211 chars


---
## 10. Ringkasan Hasil Semua Agent

Print semua fields terukur dari tiap agent dalam format ringkas.

In [22]:
DIVIDER = "=" * 65
THIN = "-" * 65

print(DIVIDER)
print("RINGKASAN HASIL SEMUA AGENT — ClarioAI (Refactored Schema)")
print(DIVIDER)

# ── Market Scout ──
ms = _agent_results.get("report_ms")
if ms:
    print(f"\n{'MARKET SCOUT':^65}")
    print(THIN)
    print(f"  demand_score      : {ms.demand_score}/10")
    print(f"  competition_level : {ms.competition_level}")
    print(f"  opportunity_score : {ms.opportunity_score}/10")
    print(f"  confidence_score  : {ms.confidence_score}/10")
    print(f"  TAM               : {ms.tam_estimate}")
    print(f"  SAM               : {ms.sam_estimate}")
    print(f"  SOM               : {ms.som_estimate}")
    print(f"  Pain Points       : {len(ms.customer_pain_points)} items")
    print(f"  Competitors       : {len(ms.competitors)} identified")
    print(f"  Opportunities     : {len(ms.validated_opportunities)} validated")
else:
    print("\n[Market Scout] Belum dijalankan")

# ── Strategic Architect ──
sa = _agent_results.get("report_sa")
if sa:
    print(f"\n{'STRATEGIC ARCHITECT':^65}")
    print(THIN)
    print(f"  confidence_score  : {sa.confidence_score}/10")
    print(f"  strengths         : {len(sa.strengths)} items")
    print(f"  weaknesses        : {len(sa.weaknesses)} items")
    print(f"  opportunities     : {len(sa.opportunities)} items")
    print(f"  threats           : {len(sa.threats)} items")
    print(f"  pestel_highlights : {len(sa.pestel_highlights)} items")
    print(f"  roadmap months    : {list(sa.roadmap.keys())}")
    print(f"  UVP               : {sa.value_proposition[:80]}..." if len(sa.value_proposition) > 80 else f"  UVP: {sa.value_proposition}")
else:
    print("\n[Strategic Architect] Belum dijalankan")

# ── Financial Analyst ──
fa = _agent_results.get("report_fa")
if fa:
    print(f"\n{'FINANCIAL ANALYST':^65}")
    print(THIN)
    print(f"  verdict           : {fa.financial_feasibility_verdict}")
    print(f"  confidence_score  : {fa.confidence_score}/10")
    print(f"  affordability     : {fa.affordability_score}/10")
    print(f"  revenue/bulan     : Rp {fa.estimated_monthly_revenue_rp:,}")
    print(f"  net_profit/bulan  : Rp {fa.estimated_monthly_net_profit_rp:,}")
    print(f"  startup_cost      : Rp {fa.estimated_startup_cost_rp:,}")
    print(f"  break_even        : {fa.estimated_break_even_months} bulan")
    print(f"  cash_runway       : {fa.cash_runway_months} bulan")
    print(f"  BEP revenue       : Rp {fa.bep_revenue_rp:,}")
    print(f"  sensitivity       : {len(fa.sensitivity_scenarios)} scenarios")
    print(f"  assumptions       : {len(fa.assumptions)} declared")
    gross_margin = 0
    total_costs = fa.estimated_monthly_cogs_rp + fa.estimated_monthly_fixed_costs_rp
    if fa.estimated_monthly_revenue_rp > 0:
        gross_margin = (fa.estimated_monthly_revenue_rp - fa.estimated_monthly_cogs_rp) / fa.estimated_monthly_revenue_rp * 100
    print(f"  gross_margin      : {gross_margin:.1f}%")
else:
    print("\n[Financial Analyst] Belum dijalankan")

# ── Ethics Agent ──
ea = _agent_results.get("report_ea")
if ea:
    print(f"\n{'ETHICS AGENT':^65}")
    print(THIN)
    print(f"  legal_risk_level  : {ea.legal_risk_level}")
    print(f"  compliance_score  : {ea.compliance_score}/10")
    print(f"  confidence_score  : {ea.confidence_score}/10")
    print(f"  complexity        : {ea.bureaucracy_complexity}")
    print(f"  duration          : {ea.bureaucracy_estimated_duration}")
    print(f"  mandatory_permits : {len(ea.mandatory_permits)} items")
    print(f"  sector_permits    : {len(ea.sector_specific_permits)} items")
    print(f"  risk_alerts       : {len(ea.critical_risk_alerts)} items")
else:
    print("\n[Ethics Agent] Belum dijalankan")

# ── Validation Layer ──
vl = _agent_results.get("report_vl")
if vl:
    print(f"\n{'VALIDATION LAYER':^65}")
    print(THIN)
    print(f"  is_valid          : {vl.is_valid}")
    print(f"  overall_confidence: {vl.overall_confidence}/10")
    critical_c = [c for c in vl.contradictions if c.severity == "critical"]
    warning_c = [c for c in vl.contradictions if c.severity == "warning"]
    print(f"  contradictions    : {len(vl.contradictions)} total ({len(critical_c)} critical, {len(warning_c)} warning)")
    print(f"  missing_assumptions: {len(vl.missing_assumptions)}")
    print(f"  unrealistic_claims: {len(vl.unrealistic_claims)}")
    print(f"  auto_flags        : {len(vl.auto_flags)}")
    for f in vl.auto_flags:
        print(f"    {f}")
else:
    print("\n[Validation Layer] Belum dijalankan")

# ── Lead Orchestrator ──
res_orch = _agent_results.get("result_orch")
if res_orch:
    print(f"\n{'LEAD ORCHESTRATOR':^65}")
    print(THIN)
    print(f"  approval_status  : {res_orch.get('approval_status')}")
    print(f"  final_verdict    : {res_orch.get('final_verdict')}")
    recs = res_orch.get('final_recommendations', [])
    print(f"  recommendations  : {len(recs)} items")
    for i, r in enumerate(recs, 1):
        print(f"    {i}. {r}")
else:
    print("\n[Lead Orchestrator] Belum dijalankan")

print()
print(DIVIDER)
print("SEMUA TEST AGENT SELESAI")
print(DIVIDER)

RINGKASAN HASIL SEMUA AGENT — ClarioAI (Refactored Schema)

                          MARKET SCOUT                           
-----------------------------------------------------------------
  demand_score      : 8/10
  competition_level : medium
  opportunity_score : 7/10
  confidence_score  : 7/10
  TAM               : Rp 2.4 Triliun/tahun
  SAM               : Rp 480 Miliar/tahun
  SOM               : Rp 1.2 Miliar/tahun (target 12 bulan pertama)
  Pain Points       : 3 items
  Competitors       : 3 identified
  Opportunities     : 3 validated

                       STRATEGIC ARCHITECT                       
-----------------------------------------------------------------
  confidence_score  : 7/10
  strengths         : 3 items
  weaknesses        : 3 items
  opportunities     : 3 items
  threats           : 3 items
  pestel_highlights : 6 items
  roadmap months    : ['month_1', 'month_2', 'month_3']
  UVP               : Kedai kopi spesialti dengan WiFi dedicated 100 Mbps dan un